In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Small test sample for v0 proof of concept
TEST_TICKERS = ["AAPL", "TSLA", "CAT", "COST", "NKE"]

def download_edgar_10k_sample(tickers):
    """Downloads 10-K filings filed strictly prior to April 2025 from SEC EDGAR."""
    os.makedirs("data/raw/edgar_tariffs", exist_ok=True)

#Map tickers to Cik via SEC API
res = request.get("https://www.sec.gov/files/company_tickers.json", headers =HEADERS)
companyindex = res.json
tickers_to_cik ={v:k for k,v in companyindex.values()}

for ticker in ticker
cik = tickers_to_cik[ticker]
if not cik:
  continue

  url = f"https://data.sec.gov/submissions/CIK{cik}.json"
  sub_res =requests.get(url, headers =HEADERS)
  sub_data = sub_res.json()
  url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        sub_res = requests.get(url, headers=HEADERS)
        if sub_res.status_code != 200:
            continue

        recent = sub_res.json()["filings"]["recent"]
        for i in range(len(recent["form"])):
            form = recent["form"][i]
            filing_date = recent["filingDate"][i]
            # Playbook Audit Rule: File date must be strictly before event window (< April 2025)
            if form == "10-K" and filing_date < "2025-04-01":
                accession = recent["accessionNumber"][i].replace("-", "")
                primary_doc = recent["primaryDocument"][i]
                doc_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{primary_doc}"

                doc_res = requests.get(doc_url, headers=HEADERS)
                if doc_res.status_code == 200:
                    filepath = f"data/raw/edgar_tariffs/{ticker}_10K_{filing_date}.html"
                    with open(filepath, "w", encoding="utf-8") as f:
                        f.write(doc_res.text)
                    print(f"[EDGAR] Saved {ticker} 10-K (Filing Date: {filing_date})")
                    break


def download_mkt_data(tickers):
  """Downloads daily price series for test tickers and market benchmark (SPY)."""
    os.makedirs("data/raw/market_data", exist_ok=True)
    symbols = tickers + ["SPY"]
    # Estimation window + Event window: April 2024 to April 2025
    df = yf.download(symbols, start="2024-04-01", end="2025-04-15")["Adj Close"]
    df.to_csv("data/raw/market_data/daily_prices.csv")
    print("[yfinance] Saved daily price series to data/raw/market_data/daily_prices.csv")

if __name__ == "__main__":
    download_edgar_10k_sample(TEST_TICKERS)
    download_market_data(TEST_TICKERS)

In [ ]:
import glob
import os
import re
import pandas as pd
from bs4 import BeautifulSoup

TARIFF_KEYWORDS = ["tariff", "tariffs", "trade policy", "duty", "duties", "customs", "trade war"]
IMPORT_KEYWORDS = ["import", "imports", "foreign supplier", "overseas manufacturing", "supply chain"]

def compute_keyword_frequency (text, keywords):
  words = re.findall(r'\w+', text.lower())
  if not words:
    return 0
    matches = sum(words.count(kw) for kw in keywords)
  return sum([1 for word in words if word in keywords]) / len(words)

def score_filings():
  files = glob.glob("data/raw/edgar_tariffs/*.html")
  os.makedirs("data/processed/scoring", exist_ok=True)

  for file in files
  filename = os.path.basename(file)
  ticker = re.search(r"(\w+)_10K", filename).group(1)
  filing_date = re.search(r"(\d{4}-\d{2}-\d{2})", filename).group(1)
  with open(file, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")
    text = soup.get_text()

    records.append({
        "ticker": ticker,
        "filing_date": filing_date,
        "tariff_freq": compute_keyword_frequency(text, TARIFF_KEYWORDS),
        "import_freq": compute_keyword_frequency(text, IMPORT_KEYWORDS)
    })

    df = pd.DataFrame(records)
  df.to_csv("data/processed/scoring/filing_scores.csv", index=False)
  print("[Scoring] Saved filing scores to data/processed/scoring/filing_scores.csv")